# Exploratory Data Analysis — Credit Card Default Dataset

**Project:** Approval Rate Prediction  
**Author:** Jose Rodrigo Carrillo Soult  
**Dataset:** UCI Credit Card Default (Taiwan, 2005) — 30,000 clients

## Business Context

In payments, **approval rate** measures the percentage of transactions
authorized successfully. Understanding what drives defaults — and therefore
declines — is critical for balancing conversion and risk.

This notebook explores the raw dataset before any modelling:
- Target distribution and class imbalance
- Feature distributions and outliers
- Correlations with the target
- Validation of engineered features

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_data
from features import build_features, PAY_COLS, BILL_COLS, PAY_AMT_COLS

# Display settings
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

DATA_PATH = "../data/raw/default of credit card clients.xls"

In [ ]:
df_raw = load_data(DATA_PATH)
df     = build_features(df_raw)

print(f"Shape (raw features):        {df_raw.shape}")
print(f"Shape (engineered features): {df.shape}")
print(f"\nApproval rate: {df['APPROVED'].mean():.2%}")
print(f"Default rate:  {1 - df['APPROVED'].mean():.2%}")

## 1. Target Distribution

The dataset has a **~78/22 class imbalance**: most clients paid on time.
This means accuracy is a misleading metric — a model that always predicts
"approved" would score 78% accuracy while being completely useless.

We use **ROC-AUC, Precision, and Recall** instead.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
counts = df["APPROVED"].value_counts().sort_index()
axes[0].bar(["Defaulted (0)", "Approved (1)"], counts.values,
            color=["#e07070", "#70a8e0"], edgecolor="white", linewidth=1.2)
axes[0].set_title("Target Distribution", fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,}\n({v/len(df):.1%})", ha="center", fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=["Defaulted", "Approved"],
            autopct="%1.1f%%", colors=["#e07070", "#70a8e0"],
            startangle=90, wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[1].set_title("Class Balance", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/01_target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Demographic Features

`SEX`, `EDUCATION`, `MARRIAGE`, and `AGE` — do they correlate with default?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

cat_features = {
    "SEX":       {1: "Male", 2: "Female"},
    "EDUCATION": {1: "Graduate", 2: "University", 3: "High school / Other"},
    "MARRIAGE":  {1: "Married", 2: "Single", 3: "Other"},
}

# --- Categorical: default rate per category ---
for ax, (col, labels) in zip(axes.flat[:3], cat_features.items()):
    rate = df.groupby(col)["APPROVED"].apply(lambda x: 1 - x.mean())
    rate.index = [labels.get(i, str(i)) for i in rate.index]
    rate.sort_values().plot(kind="barh", ax=ax, color="#70a8e0", edgecolor="white")
    ax.set_title(f"Default Rate by {col}", fontweight="bold")
    ax.set_xlabel("Default rate")
    ax.axvline(1 - df["APPROVED"].mean(), color="red", linestyle="--",
               linewidth=1, label="Overall avg")
    ax.legend(fontsize=8)

# --- AGE: histogram with default overlay ---
ax = axes[1][1]
df[df["APPROVED"] == 1]["AGE"].plot(kind="hist", ax=ax, bins=30, alpha=0.6,
                                     color="#70a8e0", label="Approved")
df[df["APPROVED"] == 0]["AGE"].plot(kind="hist", ax=ax, bins=30, alpha=0.6,
                                     color="#e07070", label="Defaulted")
ax.set_title("Age Distribution by Outcome", fontweight="bold")
ax.set_xlabel("Age")
ax.legend()

plt.tight_layout()
plt.savefig("../reports/figures/02_demographic_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Credit Limit and Amount Features

`LIMIT_BAL` and `BILL_AMT` are highly right-skewed — a small number of
clients have very high credit limits and bills. Log-transforming these
features before modelling helps linear models handle the distribution.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

plot_pairs = [
    ("LIMIT_BAL",     "LOG_LIMIT_BAL",  "Credit Limit"),
    ("BILL_AMT1",     "LOG_BILL_AMT1",  "Bill Amount (month 1)"),
]

for i, (raw_col, log_col, title) in enumerate(plot_pairs):
    # Raw
    axes[i][0].hist(df[raw_col], bins=50, color="#70a8e0", edgecolor="white", linewidth=0.5)
    axes[i][0].set_title(f"{title} — Raw", fontweight="bold")
    axes[i][0].set_xlabel("NT Dollars")

    # Log-transformed
    axes[i][1].hist(df[log_col], bins=50, color="#70ae8e", edgecolor="white", linewidth=0.5)
    axes[i][1].set_title(f"{title} — log1p", fontweight="bold")
    axes[i][1].set_xlabel("log1p(NT Dollars)")

plt.suptitle("Raw vs Log-Transformed Features", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../reports/figures/03_log_transform.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Repayment Status (PAY_0 … PAY_6)

These are the **most predictive features** in the dataset.

- `-2` = no consumption  
- `-1` = paid in full  
- `0`  = revolving credit (minimum payment)  
- `1–8` = months of payment delay  

A single month of delay (PAY_0 ≥ 1) is a strong signal of default.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default rate per PAY_0 value
pay0_rate = df.groupby("PAY_0")["APPROVED"].apply(lambda x: 1 - x.mean())
axes[0].bar(pay0_rate.index.astype(str), pay0_rate.values,
            color="#e07070", edgecolor="white")
axes[0].set_title("Default Rate by PAY_0 (most recent month)", fontweight="bold")
axes[0].set_xlabel("Repayment status")
axes[0].set_ylabel("Default rate")
axes[0].axhline(1 - df["APPROVED"].mean(), color="navy", linestyle="--",
                linewidth=1, label="Overall avg")
axes[0].legend()

# Correlation heatmap: PAY cols vs APPROVED
corr_pay = df[PAY_COLS + ["APPROVED"]].corr()["APPROVED"].drop("APPROVED").sort_values()
colors = ["#e07070" if v > 0 else "#70a8e0" for v in corr_pay.values]
axes[1].barh(corr_pay.index, corr_pay.values, color=colors, edgecolor="white")
axes[1].set_title("Correlation with Default (PAY columns)", fontweight="bold")
axes[1].set_xlabel("Pearson correlation with APPROVED")
axes[1].axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig("../reports/figures/04_repayment_status.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Engineered Features

Three groups of features were engineered from the raw columns:

| Feature group | Intuition |
|---|---|
| `UTILIZATION_1..3` | High ratio = client is near their credit limit → higher risk |
| `PAY_RATIO_1..3` | Low ratio = client pays little of their bill → higher risk |
| `MAX_DELAY` / `N_DELAYED` | Captures worst-case and frequency of late payments |
| `BILL_TREND` | Positive slope = debt is growing → higher risk |

In [ ]:
eng_features = [
    "UTILIZATION_1", "UTILIZATION_2", "UTILIZATION_3",
    "PAY_RATIO_1",   "PAY_RATIO_2",   "PAY_RATIO_3",
    "MAX_DELAY",     "N_DELAYED",     "N_DULY_PAID",
    "BILL_TREND",    "AVG_BILL",      "AVG_PAY_AMT",
]

corr_eng = df[eng_features + ["APPROVED"]].corr()["APPROVED"].drop("APPROVED").sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#e07070" if v < 0 else "#70ae8e" for v in corr_eng.values]
ax.barh(corr_eng.index, corr_eng.values, color=colors, edgecolor="white")
ax.set_title("Engineered Features — Correlation with APPROVED", fontweight="bold")
ax.set_xlabel("Pearson correlation")
ax.axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig("../reports/figures/05_engineered_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Correlation Matrix — Full Feature Setnumeric_cols = [c for c in df.select_dtypes(include=np.number).columns
                if c != "APPROVED"]
corr_matrix = df[numeric_cols + ["APPROVED"]].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle = redundant
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap="coolwarm",
    center=0, linewidths=0.3, ax=ax,
    cbar_kws={"shrink": 0.6}
)
ax.set_title("Correlation Matrix — Full Feature Set", fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("../reports/figures/06_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=np.number).columns
                if c != "APPROVED"]
corr_matrix = df[numeric_cols + ["APPROVED"]].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle = redundant
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap="coolwarm",
    center=0, linewidths=0.3, ax=ax,
    cbar_kws={"shrink": 0.6}
)
ax.set_title("Correlation Matrix — Full Feature Set", fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("../reports/figures/06_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Key Takeaways

1. **Class imbalance (~78/22)** — accuracy is not a useful metric; use ROC-AUC and F1.
2. **PAY_0 is the single strongest predictor** — one month of delay sharply increases default probability.
3. **Credit limit (LIMIT_BAL) is inversely correlated with default** — higher limits signal lower-risk clients.
4. **Bill amounts are heavily right-skewed** — log1p transforms are justified and improve model performance.
5. **Engineered features add signal** — `N_DELAYED`, `MAX_DELAY`, and `UTILIZATION_1` show meaningful correlation with the target beyond raw columns.

---
*Next notebook: `baseline_logreg.ipynb` — model training, evaluation, and business interpretation.*